# ASR Whisper-tiny (tanpa melatih)
        **audio › asr** · notebook `whisper_tiny`

        **Kapan dipakai:** Transkripsi ucapan menjadi teks tanpa data latih, multibahasa termasuk Indonesia (language='id'). Evaluasi dengan WER/CER.

        **Jebakan umum:** Klip sangat pendek bisa menghasilkan halusinasi; batasi max_new_tokens. Model tiny tidak sebaik base/small; untuk lomba pertimbangkan membawa Whisper lebih besar.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** Speech Commands mini (8 kata: down/go/left/no/right/stop/up/yes; 1200 latih / 320 uji; 1 detik, 16 kHz). Tiap klip berisi satu kata; evaluasi memakai kunci jawaban kata.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
DATA_ROOT = ROOT / "data_latihan/audio/classification/speech_commands_mini"; LANG = "en"            # "id" untuk bahasa Indonesia
N_EVAL, SEED = 160, 42
MODEL_KEY = "whisper_tiny"; OUT = out_dir("audio", "asr", MODEL_KEY); set_seed(SEED)

## 2. Data

In [ ]:
from superdb.audio import load_wavs, whisper_transcribe
te = pd.read_csv(DATA_ROOT / "test.csv").merge(pd.read_csv(DATA_ROOT / "test_labels.csv"), on="id").sample(N_EVAL, random_state=SEED).reset_index(drop=True)
Xe = load_wavs(te["path"], DATA_ROOT, 16000); truth = te["label"].str.lower().tolist(); print(Xe.shape, "| kosakata:", sorted(set(truth)))

## 3. Transkripsi (Whisper-tiny lokal, tanpa melatih)

In [ ]:
import time, re; t0 = time.time(); raw = whisper_transcribe(Xe, language=LANG); sec = time.time() - t0
norm = lambda s: re.sub(r"[^a-z]", "", s.lower())                          # buang tanda baca dan huruf besar
pred = [norm(s) for s in raw]; print(f"{sec:.1f} dtk | contoh:", list(zip(truth[:6], raw[:6])))

## 4. Evaluasi: akurasi kata, WER, CER
Pada klip satu kata, **WER = 1 − akurasi**. **CER** menghitung salah per huruf (Levenshtein).

In [ ]:
def lev(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1): prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[-1]
acc = float(np.mean([t == p for t, p in zip(truth, pred)])); cer = float(sum(lev(t, p) for t, p in zip(truth, pred)) / sum(map(len, truth)))
vocab = sorted(set(truth)); snap = [min(vocab, key=lambda v: lev(v, p)) for p in pred]               # paksa ke kosakata terdekat
acc_snap = float(np.mean([t == s for t, s in zip(truth, snap)]))
metrics = {"akurasi_kata_persis": acc, "WER": 1 - acc, "CER": cer, "akurasi_setelah_dipetakan_ke_kosakata": acc_snap, "detik": sec}; display(show_table(metrics))
wrong = [(t, r) for t, r, p in zip(truth, raw, pred) if t != p][:8]; print("contoh salah (benar → hasil):", wrong)

Kenaikan `akurasi_setelah_dipetakan_ke_kosakata` menunjukkan manfaat **pasca-pemrosesan dengan kosakata tertutup** (jika kata yang mungkin diketahui). Whisper umum bisa menulis homofon ('no' → 'know') atau kalimat sapaan pada klip pendek.

In [ ]:
save_json({"metrics": metrics, "contoh_salah": wrong}, OUT / "metrics.json"); pd.DataFrame({"benar": truth, "mentah": raw, "norm": pred}).to_csv(OUT / "transcripts.csv", index=False); print("OK. hasil di", OUT)

## Langkah berikutnya
- Audio panjang: potong per segmen (VAD) lalu transkripsi tiap segmen; metrik WER antar kalimat.
- Model lebih besar (base/small) jauh lebih akurat; bawa bobotnya bila ASR menjadi tugas utama.
- Bahasa Indonesia: `LANG = "id"` dan data uji berbahasa Indonesia.